# Vídeo, audio a R2 y subida a YouTube, desde Colab

Abrir: https://colab.research.google.com/github/DreiSup/spanishnav.al-podcast/blob/claude/quirky-dirac-x18c3n/video/video.ipynb

Para uno o varios episodios, sin pasar por tu ordenador:

- **Vídeo** (`HACER_VIDEO`): portada fija + master, en la carpeta `video/` del episodio en Drive (`docs/sop-publicacion.md`, 5 bis).
- **Audio a R2** (`SUBIR_R2`): el master a `spanishnaval/<año>/<NNN>-<slug>.mp3`, la dirección que usará el feed. **Subirlo a
  R2 no publica nada**: el episodio solo aparece en la web y en el feed cuando se añade a la web. Nunca pisa un audio que ya
  esté en R2. Necesita los Secretos de Colab `R2_ACCESS_KEY_ID` y `R2_SECRET_ACCESS_KEY` (icono de la llave, con «Acceso al
  cuaderno» activado).
- **Subida a YouTube** (`SUBIR_YOUTUBE`): el vídeo, con el título y la descripción que la sesión de dirección deja en
  `metadata.es.json` → `youtube`, programado para las 12:00 UTC de su fecha de publicación (`publicacion.fecha`), la
  misma hora a la que sale en la web y el feed. Lleva lista, categoría, idioma, «no es para niños», contenido alterado
  (voz sintética) y miniatura. Necesita los Secretos `YT_CLIENT_ID` y `YT_CLIENT_SECRET`; la primera vez pide autorizar
  la cuenta del canal y guarda la autorización en Drive (`naval-podcast/_privado/youtube_token.json`). Nunca sube dos
  veces el mismo episodio: si el canal ya tiene un vídeo con ese título, lo salta.

**Entorno: CPU** (*Entorno de ejecución → Cambiar tipo de entorno de ejecución → CPU*). No hace falta GPU.

**Uso:** pon los episodios en la celda 1 y *Entorno de ejecución → Ejecutar todas*. Drive pedirá permiso la primera vez.

Para cada episodio:

1. Comprueba en el repo que el master está **escuchado y dado por bueno** (`metadata.es.json` → `escucha`); si no,
   lo salta (se puede desactivar con `SOLO_ESCUCHADOS = False`).
2. Copia el master de Drive al disco de Colab y comprueba que se decodifica sin errores.
3. Si toca, lo sube a R2 (si ya está, comprueba que es el mismo tamaño y no lo toca) y comprueba que
   `https://audio.spanishnav.al/…` responde como `audio/mpeg` con el tamaño del master.
4. Si toca, hace el vídeo: 1920×1080, H.264, 25 fps; audio AAC-LC estéreo a 48 kHz y 192 kbps (lo que recomienda YouTube).
   Comprueba el resultado: formato, duraciones, que vídeo y audio se decodifican sin errores y que **el audio del vídeo
   es el del master** (lo compara en 12 tramos repartidos por todo el episodio).
   Lo copia a Drive.

Si un episodio falla, lo apunta y **sigue con el siguiente**. Al final, un resumen (con el tamaño y la duración de cada master, que
son los datos que necesita la web) y 30 s de cada vídeo para escucharlos aquí. No juzgues el sonido por la vista previa de Drive: recién subido, Drive lo reproduce a calidad baja.

Tarda aproximadamente un cuarto de la duración de cada episodio (7 min de audio, ~2 min).

In [ ]:
# 1. Qué episodios. Ejemplos: "004" · "006-037" · "006, 008, 010-012" · un extra sin número: "2019/extra-on-meditation"
EPISODIOS = "005-008"
HACER_VIDEO = True       # el .mp4 en video/ de Drive
SUBIR_R2 = True          # el master a R2 (no publica nada; nunca pisa un audio que ya esté)
SUBIR_YOUTUBE = True     # sube el vídeo a YouTube, programado para su fecha (necesita el vídeo hecho). False = NO sube nada a YouTube
YOUTUBE_MODO = "en_su_fecha"   # "en_su_fecha": público a las 12:00 UTC de publicacion.fecha · "privado": sin programar
SOLO_ESCUCHADOS = True   # True = solo los que en el repo están escuchados y dados por buenos
SOBRESCRIBIR = False     # True = rehace el vídeo aunque ya exista en Drive (en R2 nunca se pisa nada)

In [ ]:
# 2. Preparar: Drive, ffmpeg, la portada y las funciones
import subprocess, shutil, time, json, re, urllib.request
from pathlib import Path
import numpy as np
from google.colab import drive
drive.mount("/content/drive")

if not shutil.which("ffmpeg"):
    subprocess.run("apt-get -qq update && apt-get -qq install -y ffmpeg", shell=True, check=True)
print(subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True).stdout.splitlines()[0])

RAIZ = Path("/content/drive/MyDrive/naval-podcast/transcripts")
REPO_RAW = "https://raw.githubusercontent.com/DreiSup/spanishnav.al-podcast/claude/quirky-dirac-x18c3n"
TRABAJO = Path("/content/video_trabajo")
shutil.rmtree(TRABAJO, ignore_errors=True)
TRABAJO.mkdir()
PORTADA = TRABAJO / "portada-youtube.jpg"
subprocess.run(["curl", "-sSfL", "-o", str(PORTADA), "https://spanishnav.al/portada-youtube.jpg"], check=True)
R2_BUCKET = "spanishnaval"
R2_ENDPOINT = "https://d7303b40fa9d03af787d56d0360e6f97.r2.cloudflarestorage.com"   # no es secreto: docs/infraestructura-r2.md
AUDIO_PUBLICO = "https://audio.spanishnav.al"
R2 = None
if SUBIR_R2:
    try:
        import boto3
    except ImportError:
        subprocess.run(["pip", "-q", "install", "boto3"], check=True)
        import boto3
    from botocore.config import Config
    from google.colab import userdata
    try:
        claves = {n: userdata.get(n) for n in ("R2_ACCESS_KEY_ID", "R2_SECRET_ACCESS_KEY")}
    except Exception as e:
        raise RuntimeError("Faltan los Secretos de Colab R2_ACCESS_KEY_ID y R2_SECRET_ACCESS_KEY (icono de la llave, "
                           f"con «Acceso al cuaderno» activado), o SUBIR_R2 = False. Detalle: {type(e).__name__}: {e}")
    try:   # boto3 ≥ 1.36 calcula sumas de comprobación que R2 no necesita: solo cuando hagan falta
        cfg = Config(region_name="auto", request_checksum_calculation="when_required",
                     response_checksum_validation="when_required", retries={"max_attempts": 5})
    except TypeError:
        cfg = Config(region_name="auto", retries={"max_attempts": 5})
    R2 = boto3.client("s3", endpoint_url=R2_ENDPOINT, aws_access_key_id=claves["R2_ACCESS_KEY_ID"],
                      aws_secret_access_key=claves["R2_SECRET_ACCESS_KEY"], config=cfg)
    del claves
    # Prueba de conexión: listar unos pocos audios del bucket (el token solo puede leer y escribir objetos)
    hay = R2.list_objects_v2(Bucket=R2_BUCKET, MaxKeys=5).get("Contents", [])
    print("R2 conectado. Algunos audios del bucket:", ", ".join(o["Key"] for o in hay) or "(vacío)")

YT = None
YT_LISTA = "Naval Podcast en español"
if SUBIR_YOUTUBE:
    assert HACER_VIDEO, "SUBIR_YOUTUBE necesita HACER_VIDEO = True (si el vídeo ya está en Drive, lo coge de ahí)"
    try:
        from google_auth_oauthlib.flow import Flow
    except ImportError:
        subprocess.run(["pip", "-q", "install", "google-auth-oauthlib"], check=True)
        from google_auth_oauthlib.flow import Flow
    from google.oauth2.credentials import Credentials
    from google.auth.transport.requests import Request
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaFileUpload
    from google.colab import userdata
    ALCANCES = ["https://www.googleapis.com/auth/youtube.upload", "https://www.googleapis.com/auth/youtube"]
    TOKEN = Path("/content/drive/MyDrive/naval-podcast/_privado/youtube_token.json")
    try:
        cliente = {"installed": {"client_id": userdata.get("YT_CLIENT_ID"), "client_secret": userdata.get("YT_CLIENT_SECRET"),
                                 "auth_uri": "https://accounts.google.com/o/oauth2/auth",
                                 "token_uri": "https://oauth2.googleapis.com/token",
                                 "redirect_uris": ["http://localhost"]}}
    except Exception as e:
        raise RuntimeError("Faltan los Secretos de Colab YT_CLIENT_ID y YT_CLIENT_SECRET (con «Acceso al cuaderno»), "
                           f"o SUBIR_YOUTUBE = False. Detalle: {type(e).__name__}: {e}")
    cred = Credentials.from_authorized_user_file(str(TOKEN), ALCANCES) if TOKEN.exists() else None
    if cred and not cred.valid and cred.refresh_token:
        try:
            cred.refresh(Request())
        except Exception as e:
            print(f"La autorización guardada ya no vale ({type(e).__name__}): hay que autorizar de nuevo.")
            cred = None
    if not cred or not cred.valid:
        flujo = Flow.from_client_config(cliente, scopes=ALCANCES, redirect_uri="http://localhost:47613/")   # puerto raro: que no lo use ningún programa local
        url, _ = flujo.authorization_url(access_type="offline", prompt="consent")
        print("1. Abre este enlace y elige la cuenta del canal (si avisa de «app no verificada»: Configuración avanzada → Ir):")
        print(url)
        print("2. Al final el navegador irá a una página que no carga (localhost). Copia la dirección ENTERA de esa página.")
        respuesta = input("3. Pégala aquí y pulsa Intro: ").strip()
        flujo.fetch_token(authorization_response=respuesta.replace("http://", "https://", 1))
        cred = flujo.credentials
        TOKEN.parent.mkdir(parents=True, exist_ok=True)
        TOKEN.write_text(cred.to_json())
        print("Autorización guardada en Drive:", TOKEN)
    YT = build("youtube", "v3", credentials=cred, cache_discovery=False)
    canal = YT.channels().list(part="snippet,contentDetails", mine=True).execute()["items"][0]
    YT_SUBIDAS = canal["contentDetails"]["relatedPlaylists"]["uploads"]
    print("YouTube conectado. Canal:", canal["snippet"]["title"])
    YT_LISTA_ID = next((l["id"] for l in YT.playlists().list(part="snippet", mine=True, maxResults=50).execute().get("items", [])
                        if l["snippet"]["title"] == YT_LISTA), None)
    print("Lista:", YT_LISTA, "→", YT_LISTA_ID or "NO ENCONTRADA (los vídeos se subirán sin lista)")

VF = "scale=1920:1080:force_original_aspect_ratio=decrease,pad=1920:1080:(ow-iw)/2:(oh-ih)/2,format=yuv420p"


class Salto(Exception):
    """El episodio no se hace, pero no es un error (sin escuchar, ya hecho…)."""


def lista_de_episodios(texto):
    """'006-037', '006, 008, 010-012', '4' o '2019/extra-…' → ['006', '007', …]."""
    salida = []
    for parte in str(texto).split(","):
        parte = parte.strip()
        if not parte:
            continue
        m = re.fullmatch(r"(\d+)\s*-\s*(\d+)", parte)
        if m:
            a, b = int(m.group(1)), int(m.group(2))
            assert a <= b, f"Rango al revés: {parte}"
            salida += [f"{n:03d}" for n in range(a, b + 1)]
        elif parte.isdigit():
            salida.append(f"{int(parte):03d}")
        else:
            salida.append(parte)               # extra sin número: <año>/<carpeta>
    return list(dict.fromkeys(salida))


def sonda(ruta, campos, flujo=None):
    cmd = ["ffprobe", "-v", "error"] + (["-select_streams", flujo] if flujo else []) + \
          ["-show_entries", campos, "-of", "default=nw=1", str(ruta)]
    salida = subprocess.run(cmd, capture_output=True, text=True, check=True).stdout
    return dict(l.split("=", 1) for l in salida.strip().splitlines() if "=" in l)


def errores_al_decodificar(ruta, mapa):
    r = subprocess.run(["ffmpeg", "-v", "error", "-i", str(ruta), "-map", mapa, "-f", "null", "-"],
                       capture_output=True, text=True)
    return r.stderr.strip() or ("ffmpeg terminó con código %d" % r.returncode if r.returncode else "")


def buscar(ep):
    """Carpeta del episodio en Drive y su master."""
    if ep.isdigit():
        carpetas = sorted(p for p in RAIZ.glob(f"20*/{ep} *") if p.is_dir())
        if not carpetas:
            raise Salto("no hay carpeta en Drive")
        assert len(carpetas) == 1, f"hay {len(carpetas)} carpetas '{ep} …': {[c.name for c in carpetas]}"
        carpeta = carpetas[0]
        masters = sorted((carpeta / "audio").glob(f"{ep}-*.mp3"))
    else:
        carpeta = RAIZ / ep
        masters = sorted((carpeta / "audio").glob(f"{carpeta.name}.mp3"))
    if not masters:
        raise Salto("todavía no tiene master")
    assert len(masters) == 1, f"hay {len(masters)} masters: {[m.name for m in masters]}"
    return carpeta, masters[0]


_METADATA = {}
def metadata(anio, nombre):
    if (anio, nombre) not in _METADATA:
        try:
            with urllib.request.urlopen(f"{REPO_RAW}/episodios/{anio}/{nombre}/metadata.es.json", timeout=30) as r:
                _METADATA[anio, nombre] = json.load(r)
        except Exception as e:
            raise Salto(f"no encuentro su metadata.es.json en el repo ({e})")
    return _METADATA[anio, nombre]


def escuchado(anio, nombre):
    esc = metadata(anio, nombre).get("escucha") or {}
    return bool(esc.get("escuchado")) and esc.get("resultado") == "bueno"


def parecido(master, video, dur):
    """SNR (dB) del audio del vídeo frente al master en 12 tramos de 10 s. AAC bueno: 20–35 dB; un corte o un salto,
    cerca de 0 desde ese punto; un audio distorsionado, menos de 10 (probado el 6-nov con vídeos estropeados)."""
    def tramo(ruta, desde):
        crudo = subprocess.run(["ffmpeg", "-v", "error", "-ss", f"{desde:.3f}", "-t", "10", "-i", str(ruta),
                                "-map", "0:a:0", "-ac", "1", "-ar", "48000", "-f", "f32le", "-"],
                               capture_output=True, check=True).stdout
        return np.frombuffer(crudo, dtype=np.float32)
    snr = []
    for desde in np.linspace(1, max(1, dur - 12), 12):
        x, y = tramo(master, desde), tramo(video, desde)
        n = min(len(x), len(y))
        x, y = x[:n].astype(np.float64), y[:n].astype(np.float64)
        if n < 48000 or np.sqrt(np.mean(x ** 2)) < 1e-3:
            continue                                   # tramo demasiado corto o en silencio
        corr = np.fft.irfft(np.fft.rfft(x, 2 * n) * np.conj(np.fft.rfft(y, 2 * n)))
        lag = int(np.argmax(np.concatenate([corr[-2400:], corr[:2401]]))) - 2400   # desfase de ±50 ms como mucho
        xs, ys = (x[lag:], y[:n - lag]) if lag >= 0 else (x[:n + lag], y[-lag:])
        g = np.dot(xs, ys) / np.dot(ys, ys)
        snr.append(10 * np.log10(np.sum(xs ** 2) / np.sum((xs - g * ys) ** 2)))
    return snr


def comprobar_publico(clave, tamano):
    # Cloudflare rechaza (403) las peticiones con el User-Agent por defecto de Python: hay que identificarse
    peticion = urllib.request.Request(f"{AUDIO_PUBLICO}/{clave}", method="HEAD", headers={"User-Agent": "spanishnav-colab/1.0"})
    with urllib.request.urlopen(peticion, timeout=60) as r:
        tipo, largo = r.headers.get("Content-Type", ""), int(r.headers.get("Content-Length", -1))
    assert tipo.startswith("audio/mpeg") and largo == tamano,         f"{AUDIO_PUBLICO}/{clave} responde {tipo or '¿?'} con {largo} bytes; el master tiene {tamano}"


def subir_r2(master, anio, nombre):
    """Sube el master a R2 si no está. Nunca pisa: si ya está con otro tamaño, falla y no lo toca."""
    from botocore.exceptions import ClientError
    clave, tamano = f"{anio}/{nombre}.mp3", master.stat().st_size
    try:
        ya = R2.head_object(Bucket=R2_BUCKET, Key=clave)
    except ClientError as e:
        if e.response.get("Error", {}).get("Code") not in ("404", "NoSuchKey", "NotFound"):
            raise
        ya = None
    if ya is not None:
        assert ya["ContentLength"] == tamano,             f"en R2 ya hay {clave} con {ya['ContentLength']} bytes y el master tiene {tamano}: no se pisa, revísalo"
        comprobar_publico(clave, tamano)
        return f"ya estaba en R2 (mismo tamaño), sin tocar"
    R2.upload_file(str(master), R2_BUCKET, clave, ExtraArgs={"ContentType": "audio/mpeg"})
    assert R2.head_object(Bucket=R2_BUCKET, Key=clave)["ContentLength"] == tamano, "R2 no tiene el tamaño del master"
    for intento in range(6):                           # el dominio público puede tardar unos segundos
        try:
            comprobar_publico(clave, tamano)
            break
        except Exception:
            if intento == 5:
                raise
            time.sleep(10)
    return "subido a R2"


def titulos_del_canal():
    # Títulos de todos los vídeos del canal, también los privados y los programados (se ven porque somos el dueño)
    titulos, pagina = {}, None
    while True:
        r = YT.playlistItems().list(part="snippet", playlistId=YT_SUBIDAS, maxResults=50, pageToken=pagina).execute()
        for it in r.get("items", []):
            titulos[it["snippet"]["title"]] = it["snippet"]["resourceId"]["videoId"]
        pagina = r.get("nextPageToken")
        if not pagina:
            return titulos


def subir_youtube(video_local, anio, nombre):
    yt = metadata(anio, nombre).get("youtube") or {}
    titulo, descripcion = (yt.get("titulo") or "").strip(), (yt.get("descripcion") or "").strip()
    if not titulo or not descripcion:
        raise Salto("falta youtube.titulo o youtube.descripcion en el repo: pídeselos a la sesión de dirección")
    if yt.get("url"):
        return f"ya está en YouTube según el repo: {yt['url']}"
    ya = titulos_del_canal().get(titulo)
    if ya:
        return f"el canal ya tiene un vídeo con ese título, no se sube otra vez: https://www.youtube.com/watch?v={ya}"
    estado = {"privacyStatus": "private", "selfDeclaredMadeForKids": False, "containsSyntheticMedia": True,
              "embeddable": True}
    if YOUTUBE_MODO == "en_su_fecha":
        fecha = (metadata(anio, nombre).get("publicacion") or {}).get("fecha")
        assert fecha, "falta publicacion.fecha en el repo: sin fecha no se puede programar"
        cuando = f"{fecha}T12:00:00Z"
        if time.time() + 15 * 60 < __import__("calendar").timegm(time.strptime(cuando, "%Y-%m-%dT%H:%M:%SZ")):
            estado["publishAt"] = cuando           # programado: YouTube lo hace público a esa hora
        else:
            estado["privacyStatus"] = "public"     # su hora ya ha pasado: público ya
    cuerpo = {"snippet": {"title": titulo, "description": descripcion, "categoryId": "27",
                          "defaultLanguage": "es", "defaultAudioLanguage": "es"},
              "status": estado}
    media = MediaFileUpload(str(video_local), mimetype="video/mp4", chunksize=16 * 1024 * 1024, resumable=True)
    peticion = YT.videos().insert(part="snippet,status", body=cuerpo, media_body=media)
    respuesta = None
    while respuesta is None:
        _, respuesta = peticion.next_chunk()
    vid = respuesta["id"]
    avisos = []
    try:
        YT.thumbnails().set(videoId=vid, media_body=MediaFileUpload(str(PORTADA), mimetype="image/jpeg")).execute()
    except Exception as e:
        avisos.append(f"sin miniatura ({type(e).__name__}: {str(e)[:150]})")
    if YT_LISTA_ID:
        try:
            YT.playlistItems().insert(part="snippet", body={"snippet": {"playlistId": YT_LISTA_ID,
                                      "resourceId": {"kind": "youtube#video", "videoId": vid}}}).execute()
        except Exception as e:
            avisos.append(f"sin lista ({type(e).__name__}: {str(e)[:150]})")
    st = YT.videos().list(part="status", id=vid).execute()["items"][0]["status"]
    pedido = "programado para " + estado["publishAt"] if "publishAt" in estado else estado["privacyStatus"]
    queda = st.get("privacyStatus") + (f", se publica {st['publishAt']}" if st.get("publishAt") else "")
    YOUTUBE_SUBIDOS[f"{anio}/{nombre}"] = f"https://www.youtube.com/watch?v={vid}"
    return (f"subido: https://www.youtube.com/watch?v={vid} · pedido: {pedido} · YouTube lo deja: {queda}"
            + ("" if not avisos else " · " + " · ".join(avisos)))


YOUTUBE_SUBIDOS = {}


def procesar(ep):
    for f in list(TRABAJO.glob("*.mp3")) + list(TRABAJO.glob("*.mp4")):
        f.unlink()                                     # restos de un episodio anterior que falló
    carpeta, master_drive = buscar(ep)
    anio, nombre = carpeta.parent.name, master_drive.stem
    destino = carpeta / "video" / f"{nombre}.mp4"
    hacer_video_ahora = HACER_VIDEO and (SOBRESCRIBIR or not destino.exists())
    if not hacer_video_ahora and not SUBIR_R2 and not SUBIR_YOUTUBE:
        raise Salto("ya tiene vídeo en Drive (SOBRESCRIBIR = False)" if HACER_VIDEO else "no hay nada que hacer")
    if SOLO_ESCUCHADOS and not escuchado(anio, nombre):
        raise Salto("no está escuchado y dado por bueno en el repo")

    master = TRABAJO / master_drive.name
    shutil.copyfile(master_drive, master)
    assert master.stat().st_size == master_drive.stat().st_size, "la copia del master no tiene el tamaño del original"
    dur = float(sonda(master, "format=duration")["duration"])
    fallo = errores_al_decodificar(master, "0:a:0")
    assert not fallo, f"el master de Drive tiene errores al decodificarlo: {fallo[:500]}"
    r = {"episodio": f"{anio}/{nombre}", "bytes": master.stat().st_size, "duracion_s": round(dur), "min": dur / 60,
         "r2": "—", "video": "—", "youtube": "desactivado (SUBIR_YOUTUBE = False)" if not SUBIR_YOUTUBE else "—",
         "muestra": None}
    if SUBIR_R2:
        r["r2"] = subir_r2(master, anio, nombre)
    if not HACER_VIDEO:
        return r
    if not hacer_video_ahora:
        r["video"] = "ya tenía vídeo en Drive"
        if SUBIR_YOUTUBE:
            local = TRABAJO / destino.name
            shutil.copyfile(destino, local)
            r["youtube"] = subir_youtube(local, anio, nombre)
            local.unlink()
        return r

    salida = TRABAJO / f"{nombre}.mp4"
    proc = subprocess.run([
        "ffmpeg", "-v", "error", "-y",
        "-loop", "1", "-framerate", "25", "-i", str(PORTADA),
        "-i", str(master),
        "-map", "0:v:0", "-map", "1:a:0", "-t", f"{dur:.3f}",
        "-vf", VF, "-r", "25", "-c:v", "libx264", "-preset", "veryfast", "-tune", "stillimage", "-crf", "20",
        "-c:a", "aac", "-b:a", "192k", "-ar", "48000", "-ac", "2",
        "-movflags", "+faststart", str(salida)], capture_output=True, text=True)
    assert proc.returncode == 0 and salida.is_file(), f"ffmpeg falló: {proc.stderr[-1500:]}"

    v = sonda(salida, "stream=codec_name,width,height,pix_fmt,duration", "v:0")
    a = sonda(salida, "stream=codec_name,sample_rate,channels,duration", "a:0")
    assert (v["codec_name"], v["width"], v["height"], v["pix_fmt"]) == ("h264", "1920", "1080", "yuv420p"), f"vídeo: {v}"
    assert (a["codec_name"], a["sample_rate"], a["channels"]) == ("aac", "48000", "2"), f"audio: {a}"
    assert abs(float(v["duration"]) - dur) <= 0.5 and abs(float(a["duration"]) - dur) <= 0.5, \
        f"duraciones: master {dur:.2f} s, vídeo {v['duration']}, audio {a['duration']}"
    for mapa in ("0:v:0", "0:a:0"):
        fallo = errores_al_decodificar(salida, mapa)
        assert not fallo, f"el .mp4 tiene errores al decodificar {mapa}: {fallo[:500]}"
    snr = parecido(master, salida, dur)
    assert len(snr) >= 6, "demasiados tramos en silencio para comparar"
    assert np.median(snr) >= 20 and min(snr) >= 12, \
        f"el audio del vídeo no se parece al master (mediana {np.median(snr):.1f} dB, mínimo {min(snr):.1f} dB)"

    destino.parent.mkdir(exist_ok=True)
    temporal = destino.with_suffix(".mp4.parcial")
    shutil.copyfile(salida, temporal)
    temporal.replace(destino)
    assert destino.stat().st_size == salida.stat().st_size, "la copia en Drive no tiene el tamaño del vídeo"

    if SUBIR_YOUTUBE:
        r["youtube"] = subir_youtube(salida, anio, nombre)
    muestra = TRABAJO / f"{nombre}_muestra.wav"
    subprocess.run(["ffmpeg", "-v", "error", "-y", "-ss", str(min(60, max(0, dur - 30))), "-t", "30", "-i", str(salida),
                    "-map", "0:a:0", str(muestra)], check=True)
    master.unlink()
    salida.unlink()                                    # el disco de Colab es limitado: solo se queda la muestra
    r.update(video=f"hecho (audio {np.median(snr):.0f} dB) → {anio}/{carpeta.name}/video/{destino.name}", muestra=muestra)
    return r


LISTA = lista_de_episodios(EPISODIOS)
print(f"{len(LISTA)} episodio(s): {', '.join(LISTA)}")

In [ ]:
# 3. Procesar los episodios, uno detrás de otro (si uno falla, se apunta y se sigue)
HECHOS, SALTADOS, FALLOS = {}, {}, {}
t_total = time.time()
for i, ep in enumerate(LISTA, 1):
    t0 = time.time()
    try:
        r = procesar(ep)
        HECHOS[ep] = r
        print(f"[{i}/{len(LISTA)}] {ep}  OK  {r['min']:.1f} min · {time.time() - t0:.0f} s · R2: {r['r2']} · vídeo: {r['video']} · YouTube: {r['youtube']}")
    except Salto as e:
        SALTADOS[ep] = str(e)
        print(f"[{i}/{len(LISTA)}] {ep}  saltado: {e}")
    except Exception as e:
        FALLOS[ep] = str(e)
        print(f"[{i}/{len(LISTA)}] {ep}  FALLO: {e}")

drive.flush_and_unmount()   # espera a que Drive termine de sincronizar: no cierres el cuaderno antes
print(f"\nHechos: {len(HECHOS)} · saltados: {len(SALTADOS)} · con fallo: {len(FALLOS)} · {(time.time() - t_total) / 60:.0f} min")
for ep, motivo in FALLOS.items():
    print(f"  FALLO {ep}: {motivo}")
if HECHOS:
    print("\nDatos para la web (pásaselos a la sesión de dirección):")
    for ep, r in HECHOS.items():
        print(f"  {r['episodio']}  bytes={r['bytes']}  duracion_s={r['duracion_s']}  R2: {r['r2']}")
if HECHOS and not SUBIR_YOUTUBE:
    print("\n⚠ SUBIR_YOUTUBE = False: NINGUNO de estos episodios se ha subido a YouTube. Para subirlos, pon"
          " SUBIR_YOUTUBE = True y vuelve a ejecutar las celdas 1, 2 y 3 (el vídeo ya hecho se coge de Drive).")
if YOUTUBE_SUBIDOS:
    print("\nVídeos subidos a YouTube (pásaselos a la sesión de dirección):")
    for ep, url in YOUTUBE_SUBIDOS.items():
        print(f"  {ep}  {url}")

In [ ]:
# 4. Escuchar 30 s de cada vídeo hecho (a partir del minuto 1, sacados del .mp4 ya hecho)
from IPython.display import Audio, display
for ep, r in HECHOS.items():
    if r["muestra"]:
        print(ep, "·", r["video"])
        display(Audio(str(r["muestra"])))